Download dataset yang ingin diproses

In [1]:
!gdown 1ClaVQVepWcrw4dBesxMvoVwSdmL1THVW

Downloading...
From: https://drive.google.com/uc?id=1ClaVQVepWcrw4dBesxMvoVwSdmL1THVW
To: /content/Food Detection.v2i.yolo26.zip
100% 17.5M/17.5M [00:00<00:00, 105MB/s]


In [2]:
!gdown 1skrVBHdY0fhWKdi2zgGe1CyOmvx_7uQ2

Downloading...
From (original): https://drive.google.com/uc?id=1skrVBHdY0fhWKdi2zgGe1CyOmvx_7uQ2
From (redirected): https://drive.google.com/uc?id=1skrVBHdY0fhWKdi2zgGe1CyOmvx_7uQ2&confirm=t&uuid=24fcbdc1-0f18-4ba1-9548-045ea45d1db2
To: /content/Food_Detection_Project.v19i.yolo26.zip
100% 314M/314M [00:06<00:00, 50.2MB/s]


In [1]:
!pip install pyyaml wandb

In [2]:
!apt-get install unzip

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unzip is already the newest version (6.0-26ubuntu3.2).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [3]:
!mkdir '/content/dataset'

Unzip ke folder /dataset

In [7]:
!unzip -d "/content/dataset/Food Detection.v2i.yolo26" "/content/Food Detection.v2i.yolo26.zip"

Archive:  /content/Food Detection.v2i.yolo26.zip
  inflating: /content/dataset/Food Detection.v2i.yolo26/README.dataset.txt  
  inflating: /content/dataset/Food Detection.v2i.yolo26/README.roboflow.txt  
  inflating: /content/dataset/Food Detection.v2i.yolo26/data.yaml  
   creating: /content/dataset/Food Detection.v2i.yolo26/test/
   creating: /content/dataset/Food Detection.v2i.yolo26/test/images/
 extracting: /content/dataset/Food Detection.v2i.yolo26/test/images/1028_191030_0004_jpg.rf.3f6884ec99e0e58d7b3947b7f6efa5b6.jpg  
 extracting: /content/dataset/Food Detection.v2i.yolo26/test/images/1028_191030_0039_jpg.rf.5dbc2837eae3e82740cb39d014b8d991.jpg  
 extracting: /content/dataset/Food Detection.v2i.yolo26/test/images/1028_191030_0068_jpg.rf.e8132a3cffab2d6df1c2996c4d73610e.jpg  
 extracting: /content/dataset/Food Detection.v2i.yolo26/test/images/1028_191030_0083_jpg.rf.53f8d9917663bd6331f1d1259c7bbc91.jpg  
 extracting: /content/dataset/Food Detection.v2i.yolo26/test/images/1028_

In [8]:
!unzip -d "/content/dataset/Food_Detection_Project.v19i.yolo26" "Food_Detection_Project.v19i.yolo26.zip"

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/Food_Detection_Project.v19i.yolo26/test/labels/cd98b43f19867984_jpg.rf.e2d060e06c3a0d75739b071e5415a995.txt  
  inflating: /content/dataset/Food_Detection_Project.v19i.yolo26/test/labels/cheese_1021231240534_jpg.rf.3131ca1e2a01fb7b951405a08c7156d3.txt  
  inflating: /content/dataset/Food_Detection_Project.v19i.yolo26/test/labels/cheese_1201021021_jpg.rf.9b7c7cb40e56988dab1212a80454f5ce.txt  
  inflating: /content/dataset/Food_Detection_Project.v19i.yolo26/test/labels/cucumber1232_jpeg.rf.ece68785d38f5c3370370622a09fd9e9.txt  
  inflating: /content/dataset/Food_Detection_Project.v19i.yolo26/test/labels/dbe81f8fd25f53a7_jpg.rf.c728947b4622695a336de25e705d9cfe.txt  
  inflating: /content/dataset/Food_Detection_Project.v19i.yolo26/test/labels/download-1-_augmented_1_jpg.rf.2ee71c0a6f4ba0d99d7eb95952e77b2a.txt  
  inflating: /content/dataset/Food_Detection_Project.v19i.yolo26/test/labels/download-14-_augmented_

Atau update dataset yang sudah ada di W&B

In [5]:
# format --> wandb artifact get "FAIt (Food AI-based tracking)/<nama dataset>:latest" --root /content/dataset/<nama dataset>
#!wandb artifact get "FAIt (Food AI-based tracking)/New_Food_Dataset:latest" --root /content/dataset/New_Food_Dataset

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: Enter your choice: 2
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Downloading dataset artifact b10g004dicoding-dicoding-batch-10-g004/FAIt (Food AI-based tracking)/New_Food_Dataset:latest
wandb: Downloading large artifact 'New_Food_Dataset:latest', 311.02MB. 6244 files...
wandb:   6244 of 6244 files downloaded.  
Done. 00:00:53.3 (5.8MB/s)
wandb: Artifact downloaded to /content/dataset/New_Food_Dataset


In [6]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import yaml
from collections import OrderedDict
import shutil
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime, timezone
from google.colab import userdata
import wandb

In [7]:
def get_classnames(dataset, dataset_dir='/content/dataset/'):
  with open(dataset_dir + dataset + "/data.yaml", 'r') as file:
    read = yaml.safe_load(file)
    classnames =  read['names']
    return classnames


In [8]:


def dataset_to_df(dataset_dir, classnames, image_dir="/images/", label_dir="/labels/", image_ext=".jpg"):
  rows = []
  le = LabelEncoder()
  le.fit(classnames)

  for train_set in os.listdir(dataset_dir):
    if os.path.isdir(dataset_dir +  train_set):
      for label_file in os.listdir(dataset_dir + train_set + label_dir):
          if not label_file.endswith(".txt"):
              continue

          file_name = os.path.splitext(label_file)[0]
          label_path = os.path.join(dataset_dir + train_set + label_dir, label_file)

          image_path = os.path.join(dataset_dir + train_set + image_dir, file_name + image_ext)

          image_exists = os.path.exists(image_path)

          with open(label_path, "r") as f:
              lines = f.readlines()

          for line in lines:
              line = line.strip()

              # Skip baris kosong
              if not line:
                  continue

              parts = line.split()

              # Minimal harus ada label + bounding box
              if len(parts) < 2:
                  rows.append({
                      "label_path": label_path,
                      "label_name": None,
                      "label": None,
                      "image_path": image_path,
                      "bounding_box": line,
                      "status": 0
                  })
                  continue

              label = int(parts[0])
              label_name = le.inverse_transform([label])[0]
              bounding_box = ' '.join(parts[1:])

              rows.append({
                  "label_path": label_path,
                  "label_name": label_name,
                  "label": label,
                  "image_path": image_path,
                  "bounding_box": bounding_box,
                  "status": 1 if image_exists else 0
              })

  # Buat DataFrame
  df = pd.DataFrame(rows)
  df['label_name'] = df['label_name'].str.lower()
  df['label_name'] = df['label_name'].str.replace('_', ' ')
  return df


In [9]:
def replace_dir(filename, dest):
  fname_split = filename.split('/')
  fname_split.pop(2)
  fname_split[2] = new_dataset_dir
  dest_fname = '/'.join(fname_split)
  dir_fname = '/'.join(fname_split[:-1])
  return dest_fname, dir_fname



In [10]:
def create_dataset_dir(dataset_dir, root='/content/'):
  for train_set in ['/train', '/test', '/valid']:
    for d_type in ['/images', '/labels']:
      new_dir = os.path.join(root + dataset_dir + train_set + d_type)
      if os.path.exists(new_dir) == False:
        os.makedirs(new_dir)


In [11]:
def get_dataset_list(dataset_dir='/content/dataset/'):
  dataset_list = []
  for dataset in os.listdir(dataset_dir):
    if os.path.isdir(dataset_dir + dataset):
      dataset_list.append(dataset)
  return dataset_list


In [12]:
def all_dataset_to_df(datasets_list, dataset_dir='/content/dataset/'):
  df_list = []
  for dataset in datasets_list:
    classnames = get_classnames(dataset )
    df = dataset_to_df(dataset_dir + dataset + "/", classnames)
    df_list.append(df)
  return df_list


In [13]:
def concat_df(df_list):
  combined_df = pd.concat(df_list, axis=0, join='outer', ignore_index=True)
  return combined_df

In [14]:
def get_new_labels(df):
  labeler = LabelEncoder().fit(df['label_name'])
  df['label'] = labeler.transform(df['label_name'])
  return labeler, df

In [15]:
class FlowStyleListDumper(yaml.Dumper):
    def represent_list(self, data):
        # Represent lists in flow style (inline)
        return self.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=True)

    def represent_ordereddict(self, data):
        return self.represent_mapping('tag:yaml.org,2002:map', data)

    def represent_str(self, data): # Added to force quoting of strings
        return self.represent_scalar('tag:yaml.org,2002:str', data, style='"')


def get_yaml_file(new_dataset_dir, labeler, datasets_list, nutri_fail=[]):

  data_yml = OrderedDict({
      "train" : "../train/images",
      "val" : "../valid/images",
      "test": "../test/images",
      "nc" : len(labeler.classes_),
      "names" : labeler.classes_.tolist(),
      "additional": {
          "datasets" : datasets_list,
          "datetime" : datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S'),
          "nutrition fail" : nutri_fail

      }
  })


  # Register the custom representers
  FlowStyleListDumper.add_representer(list, FlowStyleListDumper.represent_list)
  FlowStyleListDumper.add_representer(OrderedDict, FlowStyleListDumper.represent_ordereddict)
  FlowStyleListDumper.add_representer(str, FlowStyleListDumper.represent_str) # Register for string quoting

  with open("/content/" + new_dataset_dir + "/data.yaml", 'w') as file:
      yaml.dump(data_yml, file, Dumper=FlowStyleListDumper, default_flow_style=False, sort_keys=False, width=float('inf'))

In [16]:
def scrape_nutrition_data(query_list, dataset_dir, root="/content/", base_url="https://www.fatsecret.co.id"):
    all_data = []
    base_url = "https://www.fatsecret.co.id"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    for item in query_list:
        search_url = f"{base_url}/kalori-gizi/search?q={item}"

        row = {
            'nama makanan': item,
            'kalori': '-',
            'lemak': '-',
            'karbohidrat': '-',
            'protein': '-',
            'satuan': '-',
            'link': '-',
            'status': '0'
        }

        try:
            # ======================
            # 1️⃣ Pencarian
            # ======================
            res = requests.get(search_url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

            # 🔹 Jika tidak ada hasil pencarian
            if soup.find('div', class_='searchNoResult'):
                print(f"❌ Tidak ditemukan: {item}")
                all_data.append(row)
                continue

            table = soup.find('table', class_='generic searchResult')
            if not table:
                print(f"❌ Tidak ditemukan (table kosong): {item}")
                all_data.append(row)
                continue

            first_link = table.find('a', class_='prominent')
            if not first_link:
                print(f"❌ Tidak ditemukan (link kosong): {item}")
                all_data.append(row)
                continue

            detail_url = base_url + first_link['href']

            # ======================
            # 2️⃣ Masuk halaman detail
            # ======================
            res = requests.get(detail_url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

            # ======================
            # 3️⃣ Cek link 100 gram
            # ======================
            link_100g = soup.find('a', string=lambda s: s and "100 gram" in s.lower())

            if link_100g and link_100g.get('href'):
                detail_url = base_url + link_100g['href']
                res = requests.get(detail_url, headers=headers)
                soup = BeautifulSoup(res.text, 'html.parser')

            row['link'] = detail_url

            # ======================
            # 4️⃣ Ambil satuan dari div
            # ======================
            serving_div = soup.select_one('div.serving_size.black.serving_size_value')

            if serving_div:
                row['satuan'] = serving_div.text.strip()
            else:
                row['satuan'] = '-'

            # ======================
            # 5️⃣ Ambil nilai gizi
            # ======================
            facts = soup.find_all('td', class_='fact')

            for fact in facts:
                title_el = fact.find('div', class_='factTitle')
                value_el = fact.find('div', class_='factValue')

                if title_el and value_el:
                    title = title_el.text.strip().lower()
                    value = value_el.text.strip()

                    if 'kal' in title:
                        row['kalori'] = value
                    elif 'lemak' in title:
                        row['lemak'] = value
                    elif 'karb' in title:
                        row['karbohidrat'] = value
                    elif 'prot' in title:
                        row['protein'] = value

            if facts:
                row['status'] = '1'
            else:
                print(f"⚠️ Data gizi kosong: {item}")

        except Exception as e:
            print(f"Error pada {item}: {e}")
            row['status'] = 'Error'

        all_data.append(row)
        time.sleep(1.5)

    df = pd.DataFrame(all_data)
    df.to_csv(root + dataset_dir + '/hasil_gizi_100gram.csv', index=False, encoding='utf-8-sig')
    nutri_success, nutri_fail = df[df['status'] == '1'], df[df['status'] == '0']
    return nutri_success, nutri_fail

    print("\nSelesai! Data berhasil disimpan dengan kolom satuan.")


def preprocess_query(query_list):
    cleaned_list = []
    for item in query_list:
        item = item.replace("_", " ")
        item = item.strip()
        cleaned_list.append(item)
    return cleaned_list


In [17]:
def df_to_dataset(df, new_dataset_dir, labeler, datasets_list):
  create_dataset_dir(new_dataset_dir)
  for filenames in df['label_path'].unique():
    curr_df = df.loc[df['label_path'] == filenames]
    image_path = curr_df['image_path'].values[0]

    dest_label, dir_label = replace_dir(filenames, new_dataset_dir)
    curr_df.to_csv(dest_label, sep=' ', columns=['label', 'bounding_box'], header=False, index=False)
    with open(dest_label, 'r+') as f:
        data = f.read().rstrip('\n')
        data = data.replace('"', "")
        f.seek(0)
        f.write(data)
        f.truncate()
    dest_image, dir_image = replace_dir(image_path, new_dataset_dir)
    if os.path.exists(dest_image) == False:
      shutil.copy2(image_path, dest_image)
  daftar_cari = preprocess_query(labeler.classes_.tolist())
  nutri_success, nutri_fail = scrape_nutrition_data(daftar_cari, new_dataset_dir)
  get_yaml_file(new_dataset_dir, labeler, datasets_list, nutri_fail['nama makanan'].tolist())

  return nutri_success, nutri_fail







In [18]:
class WandbHandler:
  def __init__(self, project_name, root='/content/'):
    self.project_name = project_name
    self.root = root
    wandb.login(key=userdata.get('WANDB_API_KEY'))


  def upload_to_wandb(self, dataset_dir):
    artifact = wandb.Artifact(name=dataset_dir, type="dataset")
    artifact.add_dir(self.root + dataset_dir)
    with wandb.init(project=self.project_name, job_type='upload_dataset') as run:
      run.log_artifact(artifact)

  def update_dataset(self, dataset_dir):
    with wandb.init(project=self.project_name, job_type='update_dataset') as run:
      artifact = wandb.Artifact(name=dataset_dir, type="dataset")
      artifact.add_dir(self.root + dataset_dir)
      run.log_artifact(artifact)


  def update_file(self, dataset_dir, file_name):
    with wandb.init(project=self.project_name, job_type='update_file') as run:
      saved_artifact = run.use_artifact(dataset_dir+":latest")
      draft_artifact = saved_artifact.new_draft()

      draft_artifact.remove(saved_artifact.get_entry(file_name))
      draft_artifact.add_file(local_path= self.root + dataset_dir + "/" + file_name, name=file_name)

      draft_artifact.save()





Nama dataset baru, tolong sesuaikan

In [19]:
new_dataset_dir = 'New_Food_Dataset'

In [20]:
wandb_handler = WandbHandler('FAIt (Food AI-based tracking)')

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: b10g004dicoding (b10g004dicoding-dicoding-batch-10-g004) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [21]:
datasets_list = get_dataset_list()

In [22]:
dataset_df_list = all_dataset_to_df(datasets_list)

In [23]:
combined_df = concat_df(dataset_df_list)

In [27]:
new_label, combined_df = get_new_labels(combined_df)

In [28]:
nutri_success, nutri_fail = df_to_dataset(combined_df, new_dataset_dir, new_label, datasets_list)

❌ Tidak ditemukan: cucumber


In [29]:
nutri_fail

,nama makanan,kalori,lemak,karbohidrat,protein,satuan,link,status
13,cucumber,-,-,-,-,-,-,0


In [36]:
wandb_handler.upload_to_wandb(new_dataset_dir)

wandb: Adding directory to artifact (/content/New_Food_Dataset)... Done. 5.8s


In [46]:
update = {
    13 : {
    "nama makanan": "cucumber",
    "kalori": 12,
    "lemak": "0,16g",
    "karbohidrat": "2.16g",
    "protein": "0.59g",
    "satuan": "100 gram (g)",
    "link": "https://foods.fatsecret.com/calories-nutrition/usda/cucumber-(peeled)?portionid=59108&portionamount=100.000",
    "status": 1
    }
}

In [47]:
for idx, values in update.items():
    nutri_fail.loc[idx] = values

In [48]:
nutri_fail

,nama makanan,kalori,lemak,karbohidrat,protein,satuan,link,status
13,cucumber,12,"0,16g",2.16g,0.59g,100 gram (g),https://foods.fatsecret.com/calories-nutrition...,1


In [49]:
new_nutri = concat_df([nutri_success, nutri_fail])

In [50]:
new_nutri

,nama makanan,kalori,lemak,karbohidrat,protein,satuan,link,status
0,aw cola,37,"0,02g","9,56g","0,07g",100 gram (g),https://www.fatsecret.co.id/kalori-gizi/umum/c...,1
1,beijing beef,60,2g,4g,6g,1 porsi (46 g),https://www.fatsecret.co.id/kalori-gizi/kimbo/...,1
2,black pepper rice bowl,329,"1,21g","72,54g","5,07g",100 gram (g),https://www.fatsecret.co.id/kalori-gizi/umum/r...,1
3,burger,258,"7,8g","32,1g","12,8g",1 patty (102 g),https://www.fatsecret.co.id/kalori-gizi/mcdona...,1
4,carrot,192,"9,61g","22,38g","6,27g",1 porsi,https://www.fatsecret.co.id/kalori-gizi/tuku/c...,1
5,carrot eggs,140,9g,1g,13g,1 porsi,https://www.fatsecret.co.id/kalori-gizi/mcdona...,1
6,cheese,60,5g,2g,2g,1 porsi (20 g),https://www.fatsecret.co.id/kalori-gizi/kraft/...,1
7,cheese burger,300,12g,33g,15g,1 porsi (114 g),https://www.fatsecret.co.id/kalori-gizi/mcdona...,1
8,chicken nuggets,297,"18,82g","16,32g","15,59g",100 gram (g),https://www.fatsecret.co.id/kalori-gizi/umum/c...,1
9,chicken waffle,280,"8,8g","21,8g","34,2g",1 porsi,https://www.fatsecret.co.id/kalori-gizi/shine-...,1


In [51]:
new_nutri.to_csv('/content/' + new_dataset_dir + '/hasil_gizi_100gram.csv', index=False, encoding='utf-8-sig')

In [52]:
wandb_handler.update_dataset(new_dataset_dir)

wandb: Adding directory to artifact (/content/New_Food_Dataset)... Done. 6.2s


In [53]:
get_yaml_file(new_dataset_dir, new_label, datasets_list)

In [64]:
wandb_handler.update_file(new_dataset_dir,"data.yaml")